# Figures, tables, verdict and reproducibility report

**Objective.** Rebuild every figure and table from the saved outputs, then display the scientific verdict and the Q1-readiness assessment.

**Data.** The analysis-ready PISA 2025 public-use extracts are in `data/processed/` (see `data/README.md`), so cloning the repository in Colab is enough to run the real analyses; `DEMO` switches to real data automatically. Set `DEMO = True` for a fast synthetic demonstration. **Demo outputs are not findings.** The *saved results* cells display the PISA 2025 aggregate outputs committed in `results/`.

Errors are raised, never swallowed. Every output is written to `results/` with the form name; outputs from synthetic data are prefixed `demo_`.

In [ ]:
# --- Environment (Colab or local) -------------------------------------------
REPO_URL = ""   # GitHub repository; a private repository needs a token in the URL or an uploaded copy
DEMO = None            # None = real PISA 2025 data if data/processed/ is present (it is in the repository), else synthetic; or force True/False
import os, sys, subprocess, platform
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("ability-adjusted-mcq-profiling"):
        if not REPO_URL:
            raise SystemExit("Set REPO_URL or upload the repository folder to /content.")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("ability-adjusted-mcq-profiling")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.exists("configs") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
if DEMO is None:
    DEMO = not os.path.exists("data/processed/math_formA.parquet")
import numpy, scipy, pandas, sklearn
print(platform.python_version(), "numpy", numpy.__version__, "scipy", scipy.__version__,
      "pandas", pandas.__version__, "sklearn", sklearn.__version__)
FORM = "demo" if DEMO else "math_formA"
print("Mode:", "SYNTHETIC DEMONSTRATION - not empirical results" if DEMO else "REAL PISA 2025 data")

In [ ]:
import subprocess, sys
steps = [['build_results.py'] + (['--demo'] if DEMO else []), ['build_captions.py']]
if not DEMO:
    steps += [['build_data_table.py'], ['build_crossform_table.py'], ['build_robustness_table.py'], ['build_continuous_table.py']]
for s in steps:
    print(subprocess.run([sys.executable, 'scripts/' + s[0]] + s[1:], capture_output=True, text=True).stdout[-2000:])

In [ ]:
# Every cell of the manuscript tables must equal its source CSV (prints mismatches, then the total)
if os.path.exists('manuscript'):   # the manuscript is kept outside the repository
    print(subprocess.run([sys.executable, 'scripts/check_tables.py'], capture_output=True, text=True).stdout[-3000:])

In [ ]:
# Saved PISA 2025 results (aggregate outputs committed to the repository; readable without the licensed data)
import json, pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown
def show_json(name, keys=None):
    p = Path("results/metrics") / name
    if not p.exists():
        print("missing", p); return
    d = json.loads(p.read_text(encoding="utf8"))
    print(json.dumps({k: d[k] for k in keys if k in d} if keys else d, indent=1, default=str)[:5000])
def show_csv(name):
    p = Path("results/tables") / name
    display(pd.read_csv(p)) if p.exists() else print("missing", p)
def show_fig(name):
    p = Path("results/figures") / f"{name}.png"
    display(Image(str(p))) if p.exists() else print("missing", p)

In [ ]:
show_json('verdict.json')

In [ ]:
display(Markdown(Path('reports/scientific_verdict.md').read_text(encoding='utf8')))

In [ ]:
display(Markdown(Path('reports/q1_readiness.md').read_text(encoding='utf8')))